# FotMob Match Analysis: Aston Villa vs Manchester United
### Deep Dive into Expected Goals (xG), Player Ratings & Timeline Events

This notebook demonstrates how to load, process, and visualize granular match data extracted from FotMob using the [FotMob Match Details Scraper](https://apify.com/incognito_mode/fotmob-match-details-scraper).

**Key metrics analyzed:**
- Team-level Expected Goals (xG) vs actual scoreline
- Starters and bench player ratings
- Shot volume, chances created, and defensive duels
- Unified match timeline (goals, bookings, substitutions)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization styling
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

## 1. Load Match Dataset
We load the bundled sample data for Aston Villa vs. Manchester United (`matchId: 4506541`).
To extract live matches from any FotMob URL, use `apify-client` as shown in `extract_match.py`.

In [ ]:
with open("../data/sample_match_data.json", "r", encoding="utf-8") as f:
    match_data = json.load(f)

gen = match_data["matchGeneral"]
print(f"Match: {gen['homeTeam']['name']} {gen['homeTeam']['score']} - {gen['awayTeam']['score']} {gen['awayTeam']['name']}")
print(f"League: {gen['league']} (Round {gen['round']})")
print(f"Venue: {gen['venue']} | Referee: {gen['referee']}")

## 2. Expected Goals (xG) & Team Performance Comparison
Let's compare team metrics including xG, shots, and possession.

In [ ]:
team_stats = match_data["teamStats"]
df_teams = pd.DataFrame({
    "Metric": ["xG", "Shots Total", "Shots on Target", "Possession (%)", "Big Chances"],
    "Aston Villa": [
        team_stats["expectedGoals"]["home"],
        team_stats["shotsTotal"]["home"],
        team_stats["shotsOnTarget"]["home"],
        team_stats["possessionPercent"]["home"],
        team_stats["bigChances"]["home"]
    ],
    "Manchester United": [
        team_stats["expectedGoals"]["away"],
        team_stats["shotsTotal"]["away"],
        team_stats["shotsOnTarget"]["away"],
        team_stats["possessionPercent"]["away"],
        team_stats["bigChances"]["away"]
    ]
})
df_teams

In [ ]:
# Plot Expected Goals (xG) Comparison
fig, ax = plt.subplots(figsize=(7, 4))
teams = [gen["homeTeam"]["name"], gen["awayTeam"]["name"]]
xg_values = [team_stats["expectedGoals"]["home"], team_stats["expectedGoals"]["away"]]
colors = ["#670e36", "#DA291C"]  # Club colors

bars = ax.bar(teams, xg_values, color=colors, width=0.5, edgecolor="black", linewidth=1.2)
ax.set_ylabel("Expected Goals (xG)", fontweight="bold")
ax.set_title("Match Expected Goals (xG): Aston Villa vs Man Utd", fontsize=13, fontweight="bold", pad=12)
ax.set_ylim(0, max(xg_values) * 1.3)

for bar, val in zip(bars, xg_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.08, f"{val:.2f} xG",
            ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

## 3. Player Ratings & Individual Performance
We convert the flattened `playerStats` records into a Pandas DataFrame for analysis.

In [ ]:
df_players = pd.DataFrame(match_data["playerStats"])
print(f"Total Players Tracked: {len(df_players)}")
df_players[["name", "team", "position", "rating", "minutes", "xG", "xA", "passAccuracyPct"]].head(10)

In [ ]:
# Top 10 Rated Players in the Match
top_players = df_players.sort_values(by="rating", ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(10, 6))
palette = {"Aston Villa": "#670e36", "Manchester United": "#DA291C"}
bar_colors = [palette.get(t, "gray") for t in top_players["team"]]

y_pos = range(len(top_players))
ax.barh(y_pos, top_players["rating"], color=bar_colors, edgecolor="black", linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_players["name"], fontweight="bold")
ax.set_xlabel("FotMob Match Rating", fontweight="bold")
ax.set_title("Top 10 Player Ratings (FotMob)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(6.0, 9.0)

for i, val in enumerate(top_players["rating"]):
    ax.text(val + 0.05, i, f"{val:.1f}", va="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Match Events Timeline (Goals & Substitutions)
Track when tactical substitutions were made and their impact on game momentum.

In [ ]:
df_events = pd.DataFrame(match_data["events"])
df_events

## 5. Export Processed Data
Export flattened records for integration into FPL sheets, SQL databases, or predictive models.

In [ ]:
df_players.to_csv("../data/processed_player_stats.csv", index=False)
print("[+] Successfully saved processed_player_stats.csv")